# Twitter Financial News Sentiment Dataset for SREDT Research

This notebook demonstrates the **Twitter Financial News Sentiment Dataset** artifact for the SREDT (Stochastic Risk Event Density Topology) experiment.

**What this artifact does:**
- Loads the `zeroshot/twitter-financial-news-sentiment` dataset (9,543 tweets)
- Standardizes each example to the `exp_sel_data_out` schema
- Converts integer labels (0, 1, 2) to human-readable strings: **bearish**, **neutral**, **bullish**
- Produces a structured dataset used as a validation signal for financial text understanding in SREDT

**Schema per example:**
- `input` — tweet text
- `output` — sentiment label (bearish/neutral/bullish)
- `metadata_row_index`, `metadata_task_type`, `metadata_n_classes`, `metadata_label_int`, `metadata_label_names`

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-8ea2f1-semantic-risk-evidence-development-trian/main/round-1/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded dataset: {data['metadata']['description']}")
print(f"Datasets included: {data['metadata']['datasets_included']}")
examples = data['datasets'][0]['examples']
print(f"Total examples in demo subset: {len(examples)}")

## Config

Tunable parameters for the demo. Set to minimum values for fast execution.
In the full run, `N_EXAMPLES` is set to `None` (all 9,543 examples).

In [ ]:
# Number of examples to process in the demo (None = all)
# Full run: N_EXAMPLES = None  (processes all 9,543 examples)
N_EXAMPLES = 12  # minimum: process all demo examples

## Label Mapping

Defines the integer-to-string label conversion used throughout the original script.
Labels come from the HuggingFace `zeroshot/twitter-financial-news-sentiment` dataset.

In [ ]:
TWITTER_LABEL_MAP = {0: "bearish", 1: "neutral", 2: "bullish"}

## Load and Standardize Twitter Sentiment Data

The original script reads raw JSON rows from the HuggingFace dataset and converts each row to
the `exp_sel_data_out` schema. Here we load the pre-standardized demo subset instead of the
raw dataset file, then apply the same field extraction logic to inspect the structure.

In [ ]:
def load_twitter_sentiment(raw_examples: list[dict]) -> list[dict]:
    """Load all twitter financial news sentiment rows."""
    logger.info(f"Processing {len(raw_examples)} twitter rows")
    result_examples = []
    for i, row in enumerate(raw_examples):
        text = (row.get("input") or "").strip()
        label_int = row.get("metadata_label_int")
        if not text or label_int is None:
            continue
        label_str = TWITTER_LABEL_MAP.get(int(label_int), str(label_int))
        result_examples.append({
            "input": text,
            "output": label_str,
            "metadata_row_index": row.get("metadata_row_index", i),
            "metadata_task_type": "classification",
            "metadata_n_classes": 3,
            "metadata_label_int": label_int,
            "metadata_label_names": "bearish,neutral,bullish",
        })
    logger.info(f"Built {len(result_examples)} twitter examples")
    return result_examples

# Apply N_EXAMPLES limit for demo
raw_examples = examples[:N_EXAMPLES] if N_EXAMPLES else examples
twitter_examples = load_twitter_sentiment(raw_examples)

## Build Output Structure

Assembles the final `exp_sel_data_out` schema output — the same structure written to
`full_data_out.json` by the original script.

In [ ]:
output = {
    "metadata": {
        "description": "Twitter financial news sentiment dataset for SREDT experiment (supplementary)",
        "datasets_included": ["zeroshot/twitter-financial-news-sentiment"],
    },
    "datasets": [
        {
            "dataset": "zeroshot/twitter-financial-news-sentiment",
            "examples": twitter_examples,
        },
    ],
}

size_kb = len(json.dumps(output)) / 1000
logger.info(f"Done: {len(twitter_examples)} examples, {size_kb:.1f} KB")

## Results: Sentiment Distribution and Sample Examples

Visualizes the label distribution and prints sample tweets for each sentiment class.

In [ ]:
# Count label distribution
label_counts = {"bearish": 0, "neutral": 0, "bullish": 0}
for ex in twitter_examples:
    label_counts[ex["output"]] += 1

# Print summary table
print("=" * 60)
print(f"{'TWITTER FINANCIAL SENTIMENT — DEMO RESULTS':^60}")
print("=" * 60)
print(f"\nTotal examples processed: {len(twitter_examples)}")
print(f"\n{'Label':<12} {'Count':>6} {'%':>8}")
print("-" * 30)
for label, count in label_counts.items():
    pct = 100 * count / len(twitter_examples) if twitter_examples else 0
    print(f"{label:<12} {count:>6} {pct:>7.1f}%")

# Print one sample per class
print(f"\n{'SAMPLE EXAMPLES':^60}")
print("=" * 60)
seen = set()
for ex in twitter_examples:
    lbl = ex["output"]
    if lbl not in seen:
        seen.add(lbl)
        print(f"\n[{lbl.upper()}] row {ex['metadata_row_index']}")
        print(f"  {ex['input'][:120]}")
    if len(seen) == 3:
        break

# Bar chart of label distribution
fig, ax = plt.subplots(figsize=(6, 3))
colors = {"bearish": "#e74c3c", "neutral": "#95a5a6", "bullish": "#2ecc71"}
labels = list(label_counts.keys())
counts = [label_counts[l] for l in labels]
bars = ax.bar(labels, counts, color=[colors[l] for l in labels], edgecolor="black", linewidth=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            str(count), ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Sentiment Label Distribution (Demo Subset)", fontsize=12)
ax.set_ylabel("Count")
ax.set_ylim(0, max(counts) * 1.3 if counts else 1)
plt.tight_layout()
plt.show()
print("\nSchema fields per example:", list(twitter_examples[0].keys()) if twitter_examples else "N/A")